# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iankitthakur/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row = one report date × client × content item in fact_content_daily_performance. For modelling I collapse the daily rows into one row per content page. Time window: March 2026 (month=2026-03) for development. June 2026 is the sealed test month and is not used to build the label.

In [47]:
!pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")          # the token stays in Secrets, never in the cell
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("connected")

connected


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [48]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Label:** `label_declining` = 1 if a page's impressions per day in 16–31 March fell below 80% of its rate in 1–15 March (a defined rule on observed traffic). Pages need at least 20 impressions in 1–15 March.

**Features** (all from 1–15 March, so knowable at the end of 15 March, the decision moment):

- `early_impressions`: the sum of daily impressions on days 1–15, all before the decision moment.
- `early_clicks`: the sum of daily clicks on days 1–15, all before the decision moment.
- `early_ctr`: early clicks divided by early impressions, both from days 1–15.
- `early_avg_position`: the average of the daily average position on days 1–15.
- `early_active_days`: the number of days in 1–15 with impressions, counted before the decision moment.

**Context (not features):** `client_hash_id`, `content_hash_id`, used to group rows and split train from test by client.

**Excluded on purpose:** everything after 15 March as a feature (it would leak the label), the GA4 columns (only about 4% available), and the sealed June 2026 month.

In [49]:
cols = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").fetchall()
for c in cols:
    print(c[0], "|", c[1])

report_date | DATE
client_hash_id | VARCHAR
content_hash_id | VARCHAR
client_has_gsc | BOOLEAN
client_has_ga4 | BOOLEAN
gsc_data_available | BOOLEAN
ga4_data_available | BOOLEAN
gsc_impressions | BIGINT
gsc_clicks | BIGINT
gsc_sum_position | BIGINT
gsc_avg_position | DOUBLE
ga4_pageviews | BIGINT
ga4_sessions | BIGINT
ga4_users | BIGINT
ga4_engaged_sessions | BIGINT
ga4_total_engagement_sec | BIGINT
sessions_organic | BIGINT
sessions_direct | BIGINT
sessions_referral | BIGINT
sessions_social | BIGINT
sessions_paid | BIGINT
sessions_ai | BIGINT
ai_chatgpt | BIGINT
ai_perplexity | BIGINT
ai_gemini | BIGINT
ai_copilot | BIGINT
ai_claude | BIGINT
ai_meta | BIGINT
ai_other | BIGINT
scroll_events | BIGINT
month | VARCHAR


In [50]:
con.execute(f"""
CREATE OR REPLACE VIEW march AS
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")
print("march view ready")

march view ready


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [51]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [52]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  (SELECT COUNT(*) FROM (SELECT DISTINCT report_date, client_hash_id, content_hash_id FROM march)) AS n_unique_keys
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_unique_keys
0,9841378,9841378


In [53]:
con.sql("""
SELECT
  COUNT(*) AS n_rows,
  MIN(report_date) AS first_date,
  MAX(report_date) AS last_date,
  COUNT(DISTINCT client_hash_id) AS n_clients,
  COUNT(DISTINCT content_hash_id) AS n_pages
FROM march
""").df()

,n_rows,first_date,last_date,n_clients,n_pages
0,9841378,2026-03-01,2026-03-31,55,331437


In [54]:
con.sql("""
SELECT
  COUNT(*) AS all_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_gsc_available,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_ga4_available
FROM march
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,all_rows,rows_gsc_available,rows_ga4_available
0,9841378,3611061,413966


Checks: no duplicates (9,841,378 rows and 9,841,378 unique keys). March 2026 covers 1–31 March for 55 clients and 331,437 pages. Only about 37% of rows have Search Console data and about 4% have GA4 data (checked with IS TRUE).

In [55]:
frame = con.sql("""
SELECT
  client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS early_impressions,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS early_clicks,
  SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-15') * 1.0
    / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15'), 0) AS early_ctr,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS early_avg_position,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS early_active_days,
  SUM(gsc_impressions)  FILTER (WHERE report_date >  DATE '2026-03-15') AS late_impressions
FROM march
WHERE gsc_data_available IS TRUE
  AND hash(content_hash_id) % 50 = 0
GROUP BY 1, 2
""").df()

df = frame[frame["early_impressions"] >= 20].copy()
df["late_impressions"] = df["late_impressions"].fillna(0)
df["label_declining"] = ((df["late_impressions"] / 16) < 0.8 * (df["early_impressions"] / 15)).astype(int)

features = ["early_impressions", "early_clicks", "early_ctr", "early_avg_position", "early_active_days"]
df[features] = df[features].fillna(0)

print(df.shape)
print("declining rate:", round(df["label_declining"].mean(), 3))
df[["content_hash_id"] + features + ["label_declining"]].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2191, 9)
declining rate: 0.329


,content_hash_id,early_impressions,early_clicks,early_ctr,early_avg_position,early_active_days,label_declining
0,content_f71459b346aba398,70.0,0.0,0.000000,28.862868,14,0
3,content_a6ae52d901485985,3377.0,1.0,0.000296,7.393698,15,0
4,content_cd89d1d2683c8528,550.0,0.0,0.000000,11.597114,15,0
5,content_6b6678e20915eee1,4394.0,22.0,0.005007,2.748544,15,0
6,content_671f5173daeed371,24.0,0.0,0.000000,1.375000,12,0
7,content_770560f551515fe8,97.0,0.0,0.000000,7.249259,15,1
8,content_101ef6c81d955635,3460.0,2.0,0.000578,3.975031,15,0
9,content_e56eff9f19343b60,1907.0,5.0,0.002622,7.261595,15,1
10,content_1ffc1be237ff0ce1,281.0,1.0,0.003559,5.239286,15,0
11,content_fc2bb9302b949ad9,61.0,0.0,0.000000,33.511678,14,0


In [56]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

def quick_score(cols):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
    tr, te = next(gss.split(df, groups=df["client_hash_id"]))
    m = DecisionTreeClassifier(max_depth=4, random_state=0).fit(df.iloc[tr][cols], df.iloc[tr]["label_declining"])
    p = m.predict_proba(df.iloc[te][cols])[:, 1]
    return roc_auc_score(df.iloc[te]["label_declining"], p)

print("honest AUC:", round(quick_score(features), 3))

honest AUC: 0.56


In [57]:
df["late_vs_early"] = (df["late_impressions"] / 16) / (df["early_impressions"] / 15)
print("leaky AUC:", round(quick_score(features + ["late_vs_early"]), 3))

df = df.drop(columns=["late_vs_early"])
print("honest AUC again:", round(quick_score(features), 3))

leaky AUC: 1.0
honest AUC again: 0.56


I added late_vs_early, a column built from the same future days as the label. The score jumped from 0.56 to 1.0. I deleted it and keep the honest number of 0.56. The honest score is only a little better than random (0.5), so these five early-March features carry weak signal on their own in this small sample (2,191 pages). The leaky score looked perfect only because the new column contained the answer.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Named limitation: only about 4% of March rows have GA4 data, so my slice uses Search Console signals only. Anything about on-site behaviour (sessions, engagement) is outside what this slice can tell me.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.